# Install packages

In [ ]:
!pip install matplotlib ultralytics segmentation-models-pytorch torch torchvision Pillow numpy transformers opencv-python scipy

# GASNet Model

In [ ]:
import torch
import torch.nn as nn
import torchvision.models as models
import torch.nn.functional as F

class Backbone(nn.Module):
    def __init__(self):
        super().__init__()
        mobilenet = models.mobilenet_v3_small(weights="DEFAULT")
        self.features = mobilenet.features
        # Stage indices for skip connection
        self.skip_idx = 3

    def forward(self, x):
        early_feat = None
        for i, layer in enumerate(self.features):
            x = layer(x)
            if i == self.skip_idx:
                early_feat = x    # save early feature map
        return x, early_feat  

class Head(nn.Module):
    def __init__(self, in_ch):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(in_ch, 64, 3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(64, 1, 1)
        )

    def forward(self, x):
        return self.net(x)


class Fusion(nn.Module):
    def __init__(self):
        super().__init__()
        self.attention = nn.Sequential(
            nn.Conv2d(2,8,3,padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(8,1,1),
            nn.Sigmoid()
        )

    def forward(self,feature,prior,boundary):
        prior = F.interpolate(prior,feature.shape[-2:],mode="bilinear",align_corners=False)
        boundary = F.interpolate(boundary,feature.shape[-2:],mode="bilinear",align_corners=False)
        prior_mag = torch.abs(prior)
        weight = self.attention(torch.cat([prior_mag, boundary], dim=1))

        return feature*weight


class Decoder(nn.Module):
    def __init__(self, in_ch, skip_ch):
        super().__init__()
        self.conv1 = nn.Sequential(
            nn.Conv2d(in_ch+skip_ch,128,3,padding=1),
            nn.ReLU(inplace=True)
        )

        self.conv2 = nn.Sequential(
            nn.Conv2d(128,64,3,padding=1),
            nn.ReLU(inplace=True)
        )

        self.out = nn.Conv2d(64,1,1)

    def forward(self, feature, skip, output_size):

        feature = F.interpolate(feature,scale_factor=4,mode="bilinear",align_corners=False)
        skip = F.interpolate(skip,feature.shape[-2:],mode="bilinear",align_corners=False)

        x = torch.cat([feature,skip],dim=1)
        x = self.conv1(x)
        x = self.conv2(x)
        x = self.out(x)
        x = F.interpolate(x,output_size,mode="bilinear",align_corners=False)
        
        return torch.sigmoid(x)


class GASNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.backbone = Backbone()
        self.prior_head = Head(576)
        self.boundary_head = Head(576)
        self.fusion = Fusion()
        self.decoder = Decoder(576, 24)

    def forward(self,x):
        feature,skip = self.backbone(x)
        prior = torch.tanh(self.prior_head(feature))
        boundary = torch.sigmoid(self.boundary_head(feature))
        
        fused = self.fusion(feature,prior,boundary)
        mask = self.decoder(fused,skip,x.shape[-2:])

        prior_out = F.interpolate(prior,x.shape[-2:],mode="bilinear",align_corners=False)
        boundary_out = F.interpolate(boundary,x.shape[-2:],mode="bilinear",align_corners=False)

        return {
            "mask":mask,
            "prior":prior_out,
            "boundary":boundary_out
        }

# Utility Functions

In [ ]:

import numpy as np
import cv2
import torch

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
# Normalization constants 
_MEAN = np.array([0.485, 0.456, 0.406], dtype=np.float32)
_STD  = np.array([0.229, 0.224, 0.225], dtype=np.float32)

_MORPH_KERNEL = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5, 5))
_input_tensor = torch.zeros(1, 3, 320, 320, dtype=torch.float32, device=DEVICE)

def preprocess(image_np: np.ndarray):
    """
    Optimized AMDS resize + reflection pad + normalization.
    Returns (tensor, pad_top, pad_left, new_h, new_w)
    """
    H, W = image_np.shape[:2]
    scale = 320 / max(H, W)
    new_w = int(W * scale)
    new_h = int(H * scale)

    image_r = cv2.resize(image_np, (new_w, new_h), interpolation=cv2.INTER_LINEAR)

    pad_h = 320 - new_h
    pad_w = 320 - new_w
    top   = pad_h // 2;  bottom = pad_h - top
    left  = pad_w // 2;  right  = pad_w - left

    image_p = cv2.copyMakeBorder(image_r, top, bottom, left, right, cv2.BORDER_REFLECT_101)

    # Normalized in-place to avoid a second allocation
    image_f = image_p.astype(np.float32) / 255.0
    image_f = (image_f - _MEAN) / _STD  

    # Copy into pre-allocated tensor (avoids torch.from_numpy + unsqueeze)
    _input_tensor[0].copy_(torch.from_numpy(image_f.transpose(2, 0, 1)))

    return _input_tensor, top, left, new_h, new_w


def postprocess(mask_u8: np.ndarray, min_blob_frac: float = 0.02) -> np.ndarray:
    """
    Operates on a 320x320 uint8 mask (values 0 or 255).
    Returns uint8 mask of same size.
    """
    if mask_u8.sum() == 0:
        return mask_u8

    num_labels, labels, stats, _ = cv2.connectedComponentsWithStats(
        mask_u8, connectivity=8
    )
    if num_labels <= 1:
        clean = mask_u8
    else:
        areas     = stats[1:, cv2.CC_STAT_AREA]
        total_fg  = areas.sum()
        threshold = max(1, int(total_fg * min_blob_frac))
        clean     = np.zeros_like(mask_u8)
        for label_idx, area in enumerate(areas, start=1):
            if area >= threshold:
                clean[labels == label_idx] = 255

    clean = cv2.dilate(clean, _MORPH_KERNEL)
    clean = cv2.erode(clean,  _MORPH_KERNEL)
    blurred = cv2.boxFilter(clean, ddepth=-1, ksize=(5, 5))
    _, clean = cv2.threshold(blurred, 127, 255, cv2.THRESH_BINARY)

    return clean


def run_gasnet(model, image_pil, conf_threshold=None):
    """
    Parameters
    ----------
    image_pil : PIL Image (any mode, converted to RGB internally)
    conf_threshold : float in [0,1] or None | Defaults to 0.55

    Returns
    -------
    (binary_mask np.uint8, num_detections=1) | binary_mask is 0/1 (not 0/255).     
    """
    threshold = conf_threshold if conf_threshold is not None else 0.55
    image_np = np.array(image_pil.convert("RGB"))
    H_orig, W_orig = image_np.shape[:2]

    tensor, top, left, new_h, new_w = preprocess(image_np)

    # Model inference
    with torch.inference_mode():
        output = model(tensor)
        # squeeze to (320,320), move to CPU once, stay as numpy
        pred = output["mask"].squeeze().cpu().numpy()

    # Threshold - uint8 for OpenCV ops 
    mask_320 = ((pred > threshold) * 255).astype(np.uint8)
    
    # Postprocess at 320×320 before upsampling 
    mask_320 = postprocess(mask_320)

    # Remove padding and resize to original dimensions 
    mask_crop = mask_320[top:top + new_h, left:left + new_w]
    mask_orig = cv2.resize(mask_crop, (W_orig, H_orig), interpolation=cv2.INTER_NEAREST)

    # Return 0/1 binary mask (not 0/255)
    return (mask_orig > 0).astype(np.uint8), 1



# Setup

In [ ]:
import cv2
import torch
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt
import time


def generate_mask(image_pil, mask):
    return Image.fromarray((mask * 255).astype(np.uint8)).convert("L")

def remove_background(image_pil, mask):
    img_arr = np.array(image_pil.convert("RGB"))
    alpha = (mask * 255).astype(np.uint8)
    rgba = np.dstack([img_arr, alpha])
    return Image.fromarray(rgba, "RGBA")

def composite_on_background(rgba_image, bg_color=(0, 0, 0)):
    """Flatten RGBA onto solid colour — video containers don't carry alpha."""
    bg = Image.new("RGB", rgba_image.size, bg_color)
    bg.paste(rgba_image, mask=rgba_image.split()[3])
    return bg

def run_from_path(model, image_path, conf_threshold=0.55, visualize=False, n_runs=1):
    """
    Loads an image, runs it through run_gasnet, measures inference time/FPS, then
    displays results side by side or returns (mask, background-removed).
    """
    image_pil = Image.open(image_path).convert("RGB")

    # Warm-up (untimed) — avoids counting one-off CUDA init / kernel selection.
    _ = run_gasnet(model, image_pil, conf_threshold=conf_threshold)

    if torch.cuda.is_available():
        torch.cuda.synchronize()
    start = time.perf_counter()

    for _ in range(n_runs):
        mask, num_det = run_gasnet(model, image_pil, conf_threshold=conf_threshold)

    if torch.cuda.is_available():
        torch.cuda.synchronize()
    elapsed = time.perf_counter() - start

    avg_time = elapsed / n_runs
    fps = 1.0 / avg_time if avg_time > 0 else 0.0
    print(f"Inference time: {avg_time * 1000:.2f} ms  |  FPS: {fps:.2f}  "
          f"(avg over {n_runs} run{'s' if n_runs > 1 else ''})")

    mask_img = generate_mask(image_pil, mask).convert("RGB")
    removed_bg_img = composite_on_background(remove_background(image_pil, mask), bg_color=(0, 0, 0))

    if not visualize:
        return mask_img, removed_bg_img
    
    # Visualize
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    axes[0].imshow(image_pil)
    axes[0].set_title("Original Image")
    axes[0].axis("off")

    axes[1].imshow(mask_img, cmap="gray")
    axes[1].set_title("Mask")
    axes[1].axis("off")

    axes[2].imshow(removed_bg_img)
    axes[2].set_title("Background Removed")
    axes[2].axis("off")

    fig.suptitle(f"Inference: {avg_time * 1000:.2f} ms  |  FPS: {fps:.2f}", fontsize=12)
    plt.tight_layout()
    plt.show()



# Run 

In [ ]:
import torch
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# load model
gasnet_model = GASNet()
gasnet_model.load_state_dict(
    torch.load(
        "gasnet.pt", # point to gasnet model weights 
        map_location=DEVICE
    )
)
gasnet_model.to(DEVICE)
gasnet_model.eval()

# Replace `image.jpg` with actual image
run_from_path(
    gasnet_model, 
    'image.jpg', conf_threshold=0.7, 
    visualize=True # set to True to display results
)

